<a target="_blank" href="https://colab.research.google.com/github/cerr/pyCERR-Notebooks/blob/main/03_autosegmentation/ai_seg_inference_ex1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## AI inference on images in planC

This notebook demonstrates the use of pyCERR in AI inference. planC keeps track of associations between various imaging objects. These objects can then be passed to an inference routine, e.g. image segmentation, registration, outcomes modeling. Example below demonstrates the steps involved in using image from planC as an input to segmentation model and then exporting the result to DICOM.

#### Install pyCERR

In [ ]:
%%capture
!pip install git+https://github.com/cerr/pyCERR/

#### Get path to example dicom dataset in pyCERR

#### Import dicom to planC

In [1]:
from cerr import plan_container as pc
dcm_dir = r"\\vpensmph\deasylab1\Aditi\forJosiah\inputDcm\Pt1"
planC = pc.load_dcm_dir(dcm_dir)

('ABADI^CHARLES', 'ABADI^CHARLES', '1.2.840.113745.101000.2197000.42561.3281.19234848', '1.3.46.670589.11.42189.5.0.7196.2016071217590287291', 'MR', 'MR', 0.0, 0.0, '1', '1', '1')
('ABADI^CHARLES', 'ABADI^CHARLES', '1.2.840.113745.101000.2197000.42561.3281.19234848', '1.3.46.670589.11.42189.5.0.9872.2016071218090902000', 'MR', 'MR', 0.0, 0.0, '1', '1', '1')
('ABADI^CHARLES', 'ABADI^CHARLES', '1.2.840.113745.101000.2197000.42561.3281.19234848', '1.3.6.1.4.1.9590.100.1.2.270066445160495479871114555075622059376', 'RTSTRUCT', 'RTSTRUCT', 'RTSTRUCT', 'RTSTRUCT', 'RTSTRUCT', 'RTSTRUCT', 'RTSTRUCT')


C:\Users\aptea\Miniconda3\envs\testcerr\lib\site-packages\pydicom\valuerep.py:443: UserWarning: The value length (472) exceeds the maximum length of 64 allowed for VR LO.
  warnings.warn(msg)
C:\Users\aptea\Miniconda3\envs\testcerr\lib\site-packages\cerr\dataclasses\structure.py:116: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  seg_matches = np.where(contr_sop_inst_list == scan_sop_inst)


In [2]:
import numpy as np
seriesDescriptions = np.array([s.scanInfo[0].seriesDescription for s in planC.scan])
t2ScanIndex = np.argwhere(seriesDescriptions == "Axial T2")
t2ScanIndex = t2ScanIndex[0,0]
allInds = np.arange(len(planC.scan))
allInds = np.delete(allInds, t2ScanIndex)
print("T2 scan index:")
print(t2ScanIndex)
adcScanIndex = allInds[0]
print("ADC scan index:")
print(adcScanIndex)
structNames = np.array([s.structureName for s in planC.structure])
print(structNames)
prostStructNum = np.argwhere(structNames == "CTV_PROST_DLV3")
prostStructNum = prostStructNum[0, 0]
print("Prostate structure index:")
print(prostStructNum)
from cerr.dataclasses import scan as cerrScan
prostScanNum = cerrScan.getScanNumFromUID(planC.structure[prostStructNum].assocScanUID, planC)
print(prostScanNum)

T2 scan index:
0
ADC scan index:
1
['Bladder_O_DLV3' 'CTV_PROST_DLV3' 'Penile_Bulb_DLV3' 'Rectum_O_DLV3'
 'Urethra_Foley_DLV3' 'Rectal_Spacer_DLV3' 'Bowel_Lg_DLV3']
Prostate structure index:
1
0


In [4]:
from cerr.dataclasses import structure as cerrStruct
if prostScanNum != adcScanIndex:
    planC = cerrStruct.copyToScan(prostStructNum, adcScanIndex, planC)
prostStructNumOnADC = len(planC.structure) - 1

In [6]:
from cerr.viewer import pycerr_napari as vwr
scanNum = [t2ScanIndex, adcScanIndex]
doseNum = []
strNum = [prostStructNum, prostStructNumOnADC]
displayMode = '2d' # '2d - path' or '3d - surface'
viewer, scan_layer, dose_layer, struct_layer = \
vwr.show_scan_struct_dose(scanNum, strNum, doseNum, planC, displayMode)

C:\Users\aptea\Miniconda3\envs\testcerr\lib\site-packages\napari\layers\shapes\_shapes_utils.py:627: RuntimeWarning: invalid value encountered in cast
  y = np.sign(x).astype(int)


#### Export the desired image to nii or pass numpy array to inference wrapper

In [3]:
import tempfile
import os
niiExportDir = tempfile.gettempdir()
scanNum = 0 # index of scan to export to nii
str_num = 0 # index of segmentation to export to nii
scanNiiFile = os.path.join(niiExportDir,'scan_from_cerr.nii.gz')
planC.scan[scanNum].save_nii(scanNiiFile)
str_name = planC.structure[str_num].structureName
strNiiFile = os.path.join(niiExportDir,str_name+'.nii.gz')
planC.structure[str_num].save_nii(strNiiFile,planC)
print("Exported nii files to " + niiExportDir)

Exported nii files to C:\Users\aptea\AppData\Local\Temp


#### Read segmentation output as nii

In [4]:
niiSegOutputFile = strNiiFile # same as input segmentation for now. This will be output of an AI model
assocScanNum = 0
labelsDict = {1: "test_ai_struct"}
planC = pc.load_nii_structure(niiSegOutputFile, assocScanNum, planC, labelsDict)


#### Export segmentation to dicom

In [5]:
from cerr.dcm_export import rtstruct_iod
rtstructFile = os.path.join(niiExportDir,"ai_seg_test.dcm")
structNumV = [len(planC.structure)-1] # Export the last structure in the list
seriesDescription = "AI Generated"
rtstruct_iod.create(structNumV,rtstructFile,planC,seriesDescription)


Writing RTSTRUCT file ... C:\Users\aptea\AppData\Local\Temp\ai_seg_test.dcm
File saved.


#### Clean-up exported files

In [ ]:
os.remove(scanNiiFile)
os.remove(strNiiFile)
os.remove(rtstructFile)